# CUTLASS / CuTe 主线 · 第 4/8 课：Tiled Copy、向量化与 Predication

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：设计线程—数据映射，区分逻辑边界 mask 与对齐/向量化条件。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：CUDA 线程模型、GEMM、C++ 模板基础
- 本课在路线中的作用：Tiled Copy 把 copy atom 与线程/值 layout 组合，描述哪些线程搬哪些元素；它是数据运动的可组合对象。

## 核心心智模型

### 1. 它是什么，解决什么问题

Tiled Copy 把 copy atom 与线程/值 layout 组合，描述哪些线程搬哪些元素；它是数据运动的可组合对象。

### 2. 它如何工作

CTA tile 先 partition 到线程；每线程以向量 atom 搬连续元素，边界 tile 用 identity coordinate tensor 生成 predicate。

### 3. 正确性条件与常见误区

逻辑尺寸不规则时必须 predicate；向量化还要求地址对齐和连续维长度满足，不可用 mask 修复未对齐指令。

### 4. 性能与工程取舍

更宽向量减少指令数但提高对齐约束；过多 predicate 会使尾块低效，可分主路径与尾路径。

## 图解

![Tiled Copy API 图](assets/figs/fig_05_make_tiled_copy_API.png)

请沿着本课的层级/数据流重新标注图中对象；图片只辅助建立结构，不替代代码与边界推理。


## 具体演示

N=130、每次搬 8 个 half 时，前 128 元素走完整向量，最后 2 个需 predicate 或标量尾处理。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐向量 copy 的 lane 有效掩码。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
def copy_mask(base, lanes, logical_size):
    """base 是本向量首元素，返回每个 lane 是否在逻辑边界内。"""
    # TODO：只补齐下面这个表达式。
    return ______

assert copy_mask(128, 8, 130) == [True, True, False, False, False, False, False, False]
assert all(copy_mask(0, 8, 8))


### 检查方法

运行本单元格；所有 `assert` 必须通过。另手工构造一个边界输入，解释预期结果。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“Tiled Copy、向量化与 Predication”的工作机制。

**你的答案：**


### Q2

有 predicate 就能安全执行任意未对齐的 128-bit load 吗？

**你的答案：**


### Q3

如何把规则主体与尾块拆成两个路径，何时值得？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
def copy_mask(base, lanes, logical_size):
    """base 是本向量首元素，返回每个 lane 是否在逻辑边界内。"""
    # 参考实现：表达式直接对应上文不变量。
    return [base + lane < logical_size for lane in range(lanes)]

assert copy_mask(128, 8, 130) == [True, True, False, False, False, False, False, False]
assert all(copy_mask(0, 8, 8))


### Q1 参考答案

CTA tile 先 partition 到线程；每线程以向量 atom 搬连续元素，边界 tile 用 identity coordinate tensor 生成 predicate。

### Q2 参考答案

判断时先检查本课不变量：逻辑尺寸不规则时必须 predicate；向量化还要求地址对齐和连续维长度满足，不可用 mask 修复未对齐指令。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：更宽向量减少指令数但提高对齐约束；过多 predicate 会使尾块低效，可分主路径与尾路径。

## 参考资料

- [CuTe Layout Algebra](https://docs.nvidia.com/cutlass/latest/media/docs/cpp/cute/01_layout.html)
- [CuTe Tensors](https://docs.nvidia.com/cutlass/latest/media/docs/cpp/cute/03_tensor.html)
- [CuTe Algorithms](https://docs.nvidia.com/cutlass/latest/media/docs/cpp/cute/04_algorithms.html)
- [CUTLASS GEMM API](https://docs.nvidia.com/cutlass/latest/media/docs/cpp/gemm_api.html)
- [CUTLASS repository](https://github.com/NVIDIA/cutlass)

资料用于建立事实基线；面试回答仍需用自己的语言组织。